# Démo : moteur de recommandation en deux étapes

Two-tower (TensorFlow) + index FAISS pour générer les candidats, puis classeur pour ordonner. Données : MovieLens 100K.

**Mode d'emploi** : *Exécution → Tout exécuter*. Comptez 3 à 5 minutes. La dernière cellule permet de choisir un utilisateur et de voir ses recommandations.

## 1. Installation

In [ ]:
!git clone -q https://github.com/rnnttou/moteur-recommandation-bi.git
%cd moteur-recommandation-bi
!pip install -q faiss-cpu scipy

## 2. Entraînement et évaluation (pipeline complet)

In [ ]:
from recsys.config import Config
from recsys.pipeline import run

res = run(Config(dataset="ml-100k", index_type="hnsw"))
m = res["metrics"]
m[(m.k == 10) & (m.metric.isin(["recall", "ndcg"]))].pivot(index="model", columns="metric", values="value").round(3)

## 3. Recommandations pour un utilisateur

In [ ]:
import ipywidgets as w

data, out, users = res["data"], res["output"], res["users"]
items, gcols = data.items, data.genre_names
pos = {u: i for i, u in enumerate(users)}
liked = data.interactions[data.interactions.rating >= 4].sort_values("timestamp")
test = res["splits"].test

def genres(i):
    return ", ".join(g for g in gcols if items.loc[i, g] == 1) or "?"

def show(user):
    u = data.users.loc[user]
    print(f"Utilisateur {user} : {u.gender}, {u.age} ans, {u.occupation}")
    print()
    print("Derniers films aimés (note >= 4) :")
    for i in liked[liked.user == user].item.tail(5):
        print(f"  - {items.loc[i, 'title']}  [{genres(i)}]")
    print()
    print("Top-10 recommandé par le moteur (films déjà vus exclus) :")
    test_items = set(test[test.user == user].item)
    for r, (i, s) in enumerate(zip(out["ranked"][pos[user]][:10], out["ranked_scores"][pos[user]][:10]), 1):
        flag = "  <- film réellement aimé ensuite" if i in test_items else ""
        print(f"  {r:2d}. {items.loc[i, 'title']}  [{genres(i)}]  score={s:.2f}{flag}")

w.interact(show, user=w.Dropdown(options=[int(u) for u in users[:200]], description="Utilisateur"));
